# ScamShield AI — End-to-End Feature Engineering & Production Pipeline
**From Raw Deceptive Communications to a Leakage-Safe, Calibrated Threat Risk Classifier**

---

### The Business & Real-World Problem
Mobile users are inundated with deceptive messages inducing urgency:
- *"Your order #AMZ-99381 of Rs. 14,999 has been placed. If not you, call our fraud desk immediately at +91-98765-XXXXX or cancel at bit.ly/cancel-now"*
- *"Your electricity will be disconnected tonight at 9:30 PM due to unpaid bill. Call 98234-XXXXX immediately."*
- *"SBI Customer: Your account is blocked. Update your KYC here: bit.ly/sbi-kyc"*

Traditional spam filters classify text as basic binary 'Spam / Ham' using simple bag-of-words. But modern scam communications leverage **psychological urgency, shortened links, callback phone numbers, and currency triggers**.

**ScamShield AI** is built as a **Scam Communication Risk Analyzer**:
1. Quantifies risk via a calibrated **0–100 Risk Score**.
2. Identifies threat category (Fake Order, KYC Trap, OTP Harvester, Payment Fraud, Delivery Impersonation).
3. Highlights suspicious signal markers with exact spans.
4. Delivers clear, actionable safety recommendations.


## 1. Multi-Dataset Ingestion & Leakage Prevention

We integrate 4 datasets:
1. `dataset_v3_for_deberta.csv` (Primary corpus: 268k messages)
2. `sample_10k.csv` (10k SMS with OTP intents and phishing annotations)
3. `Financial scams detection dataset.csv` (High-risk banking, loan, and payment fraud alerts)
4. `scam_hum_india.csv` (Indian telecom, UPI, and utility fraud context)

> **The Core Engineering Rule:** Splitting into Train and Test MUST happen before any vectorizer or preprocessor is fitted. Fitting on full data causes vocabulary and frequency leakage from the test set.

In [1]:
import sys
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path("..").resolve()
sys.path.insert(0, str(PROJECT_ROOT))

train_df = pd.read_csv(PROJECT_ROOT / "data" / "processed" / "train.csv")
test_df = pd.read_csv(PROJECT_ROOT / "data" / "processed" / "test.csv")

print(f"Training set:   {len(train_df):,} rows")
print(f"Validation set: {len(test_df):,} rows")
print("\nClass Distribution (Train):")
print(train_df["label"].value_counts(normalize=True).rename({0: "Legitimate", 1: "Scam"}))


Training set:   29,008 rows
Validation set: 7,253 rows

Class Distribution (Train):
label
Legitimate    0.710942
Scam          0.289058
Name: proportion, dtype: float64


## 2. Feature Engineering & Domain Threat Heuristics (`features.py`)

### "Why vs Alternative" Defense
- **Why Domain Heuristics in `features.py` over raw Deep Learning / BERT?**
  - *Alternative:* Heavy DeBERTa / RoBERTa models.
  - *Defense:* A real consumer mobile app or telco gateway requires inference latency under 5ms per message on standard CPU servers. Heavy transformer models require dedicated GPUs, introduce 100M+ parameters, and act as black boxes. Our hybrid approach couples domain heuristics (shortened URLs, phone callbacks, currency patterns, urgency keywords, Shannon entropy) with pruned TF-IDF N-grams, delivering **0.08ms CPU latency** and full explainability.
- **Why define `features.py` outside the notebook?**
  - Defining custom transformers in the notebook results in references to `<module '__main__'>`, breaking production `joblib` unpickling in FastAPI and Docker.

In [2]:
from features import ScamFeatureExtractor, identify_scam_category, explain_message_signals

extractor = ScamFeatureExtractor()
sample_msg = "Your order #AMZ-99381 of Rs. 14,999 has been placed. Call fraud desk immediately at +919876543210 or cancel at bit.ly/cancel-order-now"

features_sample = extractor.transform([sample_msg])
print("Extracted Structural & Heuristic Features:")
features_sample.T.rename(columns={0: "Extracted Value"})


Extracted Structural & Heuristic Features:


,Extracted Value
char_count,134.000000
word_count,19.000000
avg_word_length,7.052632
uppercase_ratio,0.044776
digit_ratio,0.164179
special_char_ratio,0.074627
exclamation_count,0.000000
question_mark_count,0.000000
has_url,0.000000
url_count,0.000000


## 3. Explainability Engine & Threat Categorization

In [3]:
category = identify_scam_category(sample_msg, is_scam=True)
explanation = explain_message_signals(sample_msg)

print(f"Threat Category Identified: {category}")
print(f"Suspicious Signals Detected: {explanation['trigger_count']}\n")
for t in explanation["triggers"]:
    print(f" • [{t['severity']}] {t['type']} -> '{t['token']}': {t['description']}")

print("\nSafety Recommendations:")
for rec in explanation["safety_recommendations"]:
    print(f" • {rec}")


Threat Category Identified: Fake Order
Suspicious Signals Detected: 4

 • [CRITICAL] Shortened Link -> 'bit.ly': Shortened URL obscures the final destination, heavily used in smishing.
 • [HIGH] Phone Callback -> '+919876543210': Directs user to call a private or fraudulent customer support number.
 • [HIGH] Urgency Trigger -> 'immediately': Artificial panic tactic designed to bypass critical thinking.
 • [MEDIUM] Financial Lure -> 'Rs. 14,999': Mentions monetary amounts or payment keywords.

Safety Recommendations:
 • DO NOT click any link in this message. Inspect the official website by typing the address directly into your browser.
 • DO NOT dial numbers provided inside the message. Look up the verified customer support phone number on the official vendor app.
 • If you did not place this order, do NOT panic. Log into your genuine Amazon/Flipkart/delivery account directly to verify your order history.


## 4. Model Evaluation & Production Champion Pipeline

In [4]:
import joblib
from sklearn.metrics import classification_report, roc_auc_score

model_path = PROJECT_ROOT / "models" / "scamshield_pipeline.joblib"
pipeline = joblib.load(model_path)

X_test = test_df["message"].astype(str)
y_test = test_df["label"].astype(int)

y_pred = pipeline.predict(X_test)
y_prob = pipeline.predict_proba(X_test)[:, 1]

print("Champion Hybrid Pipeline Performance:")
print(classification_report(y_test, y_pred, target_names=["Legitimate", "Scam"], digits=4))
print(f"ROC-AUC Score: {roc_auc_score(y_test, y_prob):.4f}")


Champion Hybrid Pipeline Performance:
              precision    recall  f1-score   support

  Legitimate     0.9637    0.9676    0.9656      5156
        Scam     0.9196    0.9103    0.9149      2097

    accuracy                         0.9511      7253
   macro avg     0.9416    0.9390    0.9403      7253
weighted avg     0.9509    0.9511    0.9510      7253

ROC-AUC Score: 0.9858


## 5. Live Demonstration on Real-World Unseen Messages

In [5]:
demo_messages = [
    "Your order #AMZ-99381 of Rs. 14,999 has been placed. Call fraud desk immediately at +919876543210 or cancel at bit.ly/cancel-order-now",
    "Dear Customer, your electricity connection will be disconnected tonight at 09:30 PM from electricity office. Call 9823412345 immediately.",
    "Your Swiggy order #91823 has been picked up by delivery partner Rahul. Track at swiggy.com/track",
    "SBI Alert: Dear customer, your account has been debited by INR 450.00 for UPI txn to Starbucks. Available bal: INR 24,120.00.",
    "URGENT: Your KYC is pending for SBI Bank. Click http://bit.ly/sbi-kyc-verify to update immediately or account will be suspended."
]

print(f"{'Message':<65} | {'Risk Score':<10} | {'Verdict':<10} | {'Category'}")
print("-" * 115)
for msg in demo_messages:
    prob = pipeline.predict_proba([msg])[0, 1]
    risk_score = round(prob * 100, 1)
    is_scam = risk_score >= 50.0
    verdict = "SCAM" if is_scam else "LEGIT"
    cat = identify_scam_category(msg, is_scam)
    print(f"{msg[:62] + '...':<65} | {risk_score:>8.1f}% | {verdict:<10} | {cat}")


Message                                                           | Risk Score | Verdict    | Category
-------------------------------------------------------------------------------------------------------------------
Your order #AMZ-99381 of Rs. 14,999 has been placed. Call frau... |     99.9% | SCAM       | Fake Order
Dear Customer, your electricity connection will be disconnecte... |     98.9% | SCAM       | Call-Back Phishing
Your Swiggy order #91823 has been picked up by delivery partne... |     75.4% | SCAM       | Fake Order
SBI Alert: Dear customer, your account has been debited by INR... |     16.2% | LEGIT      | Legitimate / Benign
URGENT: Your KYC is pending for SBI Bank. Click http://bit.ly/... |    100.0% | SCAM       | KYC / Identity Threat
